In [8]:
import numpy as np

def CalcCTransMatrix(
    SoSWater=1500.0,
    ComplexLongSOS=2200 * (1 - 1j * 0.005),
    ComplexShearSOS=1500 * (1 - 1j * 0.05),
    omega=270e3 * 2 * np.pi,
    ThetaIncident=0.0,
    Distance=8e-3,
    Density=1800.0,
):
    """
    Python/Numpy version of CalcCTransMatrix
    """

    # Wave numbers
    LongWaveNumber = omega / ComplexLongSOS
    ShearWaveNumber = omega / ComplexShearSOS

    Sigma = np.sin(ThetaIncident) * SoSWater

    # Initialize matrix (complex)
    CTrans = np.zeros((4, 4), dtype=np.complex128)

    beta = np.sqrt(ShearWaveNumber**2 - Sigma**2)
    alpha = np.sqrt(LongWaveNumber**2 - Sigma**2)

    kappa = ShearWaveNumber

    

    # --- Row 1 ---
    CTrans[0, 0] = -(
        (2*np.cos(beta*Distance) - 2*np.cos(alpha*Distance))*Sigma**2
        - kappa**2*np.cos(beta*Distance)
    ) / kappa**2

    CTrans[0, 1] = -(
        2j*np.sin(alpha*Distance)*Sigma**3
        + (2j*alpha*beta*np.sin(beta*Distance)
           - 1j*kappa**2*np.sin(alpha*Distance))*Sigma
    ) / (alpha*kappa**2)

    CTrans[0, 2] = (
        (np.cos(beta*Distance) - np.cos(alpha*Distance))*Sigma
    ) / (Density*omega)

    CTrans[0, 3] = -(
        1j*np.sin(alpha*Distance)*Sigma**2
        + 1j*alpha*beta*np.sin(beta*Distance)
    ) / (alpha*Density*omega)

    # --- Row 2 ---
    CTrans[1, 0] = (
        2j*np.sin(beta*Distance)*Sigma**3
        + (2j*alpha*beta*np.sin(alpha*Distance)
           - 1j*kappa**2*np.sin(beta*Distance))*Sigma
    ) / (beta*kappa**2)

    CTrans[1, 1] = (
        (2*np.cos(beta*Distance) - 2*np.cos(alpha*Distance))*Sigma**2
        + kappa**2*np.cos(alpha*Distance)
    ) / kappa**2

    CTrans[1, 2] = -(
        1j*np.sin(beta*Distance)*Sigma**2
        + 1j*alpha*beta*np.sin(alpha*Distance)
    ) / (beta*Density*omega)

    CTrans[1, 3] = CTrans[0, 2]

    # --- Row 3 ---
    CTrans[2, 0] = -(
        (4*Density*np.cos(beta*Distance)
         - 4*Density*np.cos(alpha*Distance))*omega*Sigma**3
        + (2*kappa**2*Density*np.cos(alpha*Distance)
           - 2*kappa**2*Density*np.cos(beta*Distance))*omega*Sigma
    ) / kappa**4

    CTrans[2, 1] = -(
        4j*Density*np.sin(alpha*Distance)*omega*Sigma**4
        + (4j*alpha*beta*Density*np.sin(beta*Distance)
           - 4j*kappa**2*Density*np.sin(alpha*Distance))*omega*Sigma**2
        + 1j*kappa**4*Density*np.sin(alpha*Distance)*omega
    ) / (alpha*kappa**4)

    CTrans[2, 2] = CTrans[1, 1]
    CTrans[2, 3] = CTrans[0, 1]

    # --- Row 4 ---
    CTrans[3, 0] = -(
        4j*Density*np.sin(beta*Distance)*omega*Sigma**4
        + (4j*alpha*beta*Density*np.sin(alpha*Distance)
           - 4j*kappa**2*Density*np.sin(beta*Distance))*omega*Sigma**2
        + 1j*kappa**4*Density*np.sin(beta*Distance)*omega
    ) / (beta*kappa**4)

    CTrans[3, 1] = CTrans[2, 0]
    CTrans[3, 2] = CTrans[1, 0]
    CTrans[3, 3] = CTrans[0, 0]

    return CTrans

In [9]:

freq=200e3
attenuation_solid=0.1151277918 * 0.1 * freq/1000  #Np/m
print('attenuation_solid',attenuation_solid)
c_l=2600.0
Density=1190
Distance=25.4e-3
omega=freq*2*np.pi
rho_w=1000.0
SoSWater=1500.0


c_lc= omega/(omega/c_l + 1j*attenuation_solid)
LongWaveNumber=omega/c_lc

Theta=0.0
sigma = LongWaveNumber*np.sin(Theta)
alpha=np.sqrt(LongWaveNumber**2-sigma**2)

Z1 = rho_w*omega/alpha

CTrans=CalcCTransMatrix(ComplexLongSOS=c_lc,
                        SoSWater=SoSWater,
                        omega=omega,
                        Distance=Distance,
                        ThetaIncident=Theta,
                        Density=Density)

M22 = CTrans[1,1] -CTrans[1,0]*CTrans[3,1]/CTrans[3,0]
M23 = CTrans[1,2] -CTrans[1,0]*CTrans[3,2]/CTrans[3,0]
M32 = CTrans[2,1] -CTrans[2,0]*CTrans[3,1]/CTrans[3,0]
M33 = CTrans[2,2] -CTrans[2,0]*CTrans[3,2]/CTrans[3,0]
Tr_coeff = 2*Z1/((M22+Z1*M23)*Z1 + M32 + Z1*M33)
np.sqrt(np.abs(Tr_coeff*np.conjugate(Tr_coeff))),np.log10(np.abs(Tr_coeff))*20,# np.exp(-attenuation_solid*6.5e-3)

attenuation_solid 2.3025558360000002


(np.float64(0.9413593382751618), np.float64(-0.5248912967963272))

In [10]:
freq=650e3
attenuation_solid=0.0
print('attenuation_solid',attenuation_solid)
c_l=967.0
Density=700
Distance=5e-3
omega=freq*2*np.pi
rho_w=1000.0
SoSWater=1500.0


c_lc= omega/(omega/c_l + 1j*attenuation_solid)
LongWaveNumber=omega/c_lc

Theta=0.0
sigma = LongWaveNumber*np.sin(Theta)
alpha=np.sqrt(LongWaveNumber**2-sigma**2)

Z1 = rho_w*omega/alpha

CTrans=CalcCTransMatrix(ComplexLongSOS=c_lc,
                        SoSWater=SoSWater,
                        omega=omega,
                        Distance=Distance,
                        ThetaIncident=Theta,
                        Density=Density)

M22 = CTrans[1,1] -CTrans[1,0]*CTrans[3,1]/CTrans[3,0]
M23 = CTrans[1,2] -CTrans[1,0]*CTrans[3,2]/CTrans[3,0]
M32 = CTrans[2,1] -CTrans[2,0]*CTrans[3,1]/CTrans[3,0]
M33 = CTrans[2,2] -CTrans[2,0]*CTrans[3,2]/CTrans[3,0]
Tr_coeff = 2*Z1/((M22+Z1*M23)*Z1 + M32 + Z1*M33)
np.sqrt(np.abs(Tr_coeff*np.conjugate(Tr_coeff))),np.log10(np.abs(Tr_coeff))*20,# np.exp(-attenuation_solid*6.5e-3)

attenuation_solid 0.0


(np.float64(0.9631252257853331), np.float64(-0.32634484251778706))